# Notebook 3: Clean and Grid Tree Census Data

This is the third notebook in the sequence for the HRS Botany project.

In it we will clean and merge the two tree census datasets (FERP and OFO), in preparation for gridding by EnMAP pixels.


__Notebook Inputs:__
- OFO Trees Dataset (_ofo_ground-reference_trees.gpkg_)
- FERP Data (_FERP123merged_20231029.csv_)

__Notebook Outputs:__
- Cleaned and merged Tree Dataset (_data/trees.csv_)


In [1]:
# imports

import pandas as pd
import geopandas as gpd

from hrs_botany.data_utils import load_ferp_species_table

## OFO Trees Dataset

In [2]:
ofo_trees_df = gpd.read_file('../../data/ofo/ofo_ground-reference_trees.gpkg')
plots_df = gpd.read_file('./data/plots.csv')

Let's restrict the ofo trees dataframe to the plots we selected in notebook 0.

In [3]:
ofo_plot_l = list(plots_df.plot_id[:-3])

ofo_trees_df = ofo_trees_df[ofo_trees_df.plot_id.isin(ofo_plot_l)].reset_index(drop=True)

In [4]:
print('Number of records across all selected OFO plots:', len(ofo_trees_df))

Number of records across all selected OFO plots: 34471


The OFO tree dataset specifies the tree species by its USDA PLANTS symbol. To standardize which species code we are using across datasets, we will convert these to GBIF Taxon IDs. 

First, we will convert the species_code to scientific name.

In [5]:
# 1. Load the USDA PLANTS symbols file
usda = pd.read_csv(
    "./data/usda/usda_plants_codes.txt",
    low_memory=False,
    dtype=str,
    skipinitialspace=True
)

# 2. Keep only the primary (non‐synonym) rows
primary = usda[usda["Synonym Symbol"].isna() | (usda["Synonym Symbol"] == "")].copy()

# 3. Optional: trim whitespace
for col in ["Symbol","Scientific Name with Author","Common Name","Family"]:
    primary[col] = primary[col].str.strip()

# 4. Build a lookup table
lookup = primary[[
    "Symbol",
    "Scientific Name with Author",
    "Common Name",
    "Family"
]].rename(columns={
    "Symbol":                      "species_code",
    "Scientific Name with Author": "scientific_name",
    "Common Name":                 "common_name",
    "Family":                      "family"
})

# 5. First pass merge into your tree_df
ofo_trees_df = ofo_trees_df.merge(
    lookup,
    on="species_code",
    how="left"
)

# 6. Inspect any unmatched codes
missing = ofo_trees_df[ofo_trees_df["scientific_name"].isna()]["species_code"].unique()
if len(missing):
    print("No USDA match for:", missing)
else:
    print("All codes matched!")

# 7. Hand-curated scientific_name overrides for leftover species_codes:
manual_matches = {
    "RHPU":    {"scientific_name": "Frangula purshiana"},
    "ARCPRI":  {"scientific_name": "Arctostaphylos pringlei"},
    "UNKSNAG": {"scientific_name": "Tsuga mertensiana"},
    "FRACAL":  {"scientific_name": "Frangula californica"},
    "AB":      {"scientific_name": "Fagus grandifolia"},
    "PIPJ":    {"scientific_name": "Picea pungens"},
    "CEAPAL":  {"scientific_name": "Ceanothus palmeri"},
    "QUEXMO":  {"scientific_name": "Quercus xmorehus"},
    "RHOOCC":  {"scientific_name": "Rhododendron occidentale"},
    "ARCGLA":  {"scientific_name": "Arctostaphylos glandulosa"},
    "PI":      {"scientific_name": "Picea spp."},
    "LONSUB":  {"scientific_name": "Lonicera subspicata"},
    "QUEV":    {"scientific_name": "Quercus spp."},
    "ARCPUN":  {"scientific_name": "Arctostaphylos pungens"},
    "UNK":     {"scientific_name": "Unknown"},
}

# 8. Build a tiny DataFrame of those overrides
manual_df = (
    pd.DataFrame.from_dict(manual_matches, orient="index")
      .reset_index()
      .rename(columns={"index": "species_code"})
)

# 9. Merge into ofo_trees_df, filling only where the lookup left blanks
ofo_trees_df = (
    ofo_trees_df
    .merge(manual_df, on="species_code", how="left", suffixes=("", "_manual"))
)

# 10. For scientific_name, fall back to the manual value if it was missing
ofo_trees_df["scientific_name"] = ofo_trees_df["scientific_name"].fillna(
    ofo_trees_df["scientific_name_manual"]
)
ofo_trees_df.drop(columns="scientific_name_manual", inplace=True)

# 11. Final sanity check
still_missing = ofo_trees_df.loc[
    ofo_trees_df["scientific_name"].isna(), "species_code"
].unique()
if len(still_missing):
    print("Still no scientific_name for:", still_missing)
else:
    print("All species_codes now have scientific_name!")


# Now tree_df has:
#  - usda_symbol
#  - scientific_name
#  - common_name
#  - family
#  - fia_code


No USDA match for: ['PI' 'UNKSNAG' 'QUEV' 'PIPJ' 'AB' 'UNK' 'ARCPRI' 'FRACAL' 'CEAPAL'
 'RHOOCC' 'ARCGLA' 'LONSUB' 'ARCPUN' 'QUEXMO' 'RHPU']
All species_codes now have scientific_name!


We will now join in the GBIF Taxon ID. 

And there are a few typos in the scientific names: we will now manually fix them.

In [6]:
import pandas as pd
from pygbif import species
import time

# 1. Copy and normalize your scientific names

ofo_trees_df['sciname'] = (
    ofo_trees_df['scientific_name']
      .str.strip()
      .str.split()
      .str[:2]
      .str.join(' ')
      .str.lower()
)

# 2. Prepare a cache dict + manual overrides for the few bad names
taxon_cache = {}
name_overrides = {
    'quercus ×deamii': 'quercus deamii',  # drop the hybrid sign
    'salix l.':          'Salix L.',         # look up the genus  
    'unknown':           None             # skip entirely
}

# 3. Iterate unique names and query GBIF (with overrides)
for name in ofo_trees_df['sciname'].dropna().unique():
    if name in name_overrides:
        override = name_overrides[name]
        # if override is None, we know it’s “unknown” → leave as None
        if override is None:
            taxon_cache[name] = None
            continue
        lookup_name = override
    else:
        lookup_name = name

    try:
        res = species.name_backbone(name=lookup_name)
        key = res.get('usageKey') or None
    except Exception:
        key = None

    taxon_cache[name] = key
    time.sleep(0.1)  # be polite to GBIF

# 4. Map back onto your DataFrame
ofo_trees_df['gbif_taxon_key'] = ofo_trees_df['sciname'].map(taxon_cache)

# 5. Check what still didn’t resolve
unmapped = sorted(x for x, k in taxon_cache.items() if k is None)
print(f"Unmapped names ({len(unmapped)}):", unmapped)


Unmapped names (1): ['unknown']


In [39]:
live_ofo_tree_df = ofo_trees_df[(ofo_trees_df.live_dead == 'L') & (ofo_trees_df.growth_form == 'tree')]
print('Number of live trees:', len(live_ofo_tree_df))
print('Percent live trees missing height:', sum(live_ofo_tree_df.height.isna())/len(live_ofo_tree_df))
print('Percent live trees missing dbh:', sum(live_ofo_tree_df.dbh.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_position:', sum(live_ofo_tree_df.crown_position.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_ratio:', sum(live_ofo_tree_df.crown_ratio.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_ratio_compacted:', sum(live_ofo_tree_df.crown_ratio_compacted.isna())/len(live_ofo_tree_df))
print('Percent live trees missing height_to_crown:', sum(live_ofo_tree_df.height_to_crown.isna())/len(live_ofo_tree_df))
print('Percent live trees missing ohvis:', sum(live_ofo_tree_df.ohvis.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_width_1:', sum(live_ofo_tree_df.crown_width_1.isna())/len(live_ofo_tree_df))
print('Percent live trees missing crown_width_2:', sum(live_ofo_tree_df.crown_width_2.isna())/len(live_ofo_tree_df))

Number of live trees: 27860
Percent live trees missing height: 0.6198851399856425
Percent live trees missing dbh: 0.059547738693467335
Percent live trees missing crown_position: 0.7582555635319455
Percent live trees missing crown_ratio: 1.0
Percent live trees missing crown_ratio_compacted: 1.0
Percent live trees missing height_to_crown: 0.9917803302225413
Percent live trees missing ohvis: 0.7582555635319455
Percent live trees missing crown_width_1: 0.9917803302225413
Percent live trees missing crown_width_2: 0.9917803302225413


In [ ]:
ofo_cols = ['plot_id', 'gbif_taxon_key', 'geometry', 'scientific_name', 'common_name', 'sciname' 'height', 'dbh', 'crown_position', 'ohvis']

ofo_trees_df = ofo_trees_df[ofo_cols]


,subplot_id,height,height_allometric,height_above_plot_center,dbh,crown_position,ohvis,crown_ratio,crown_ratio_compacted,height_to_crown,...,crown_width_2,crown_width_allometric,decay_class,damage_1,damage_2,damage_3,damage_4,damage_5,notes,gbif_taxon_key
count,6624.000000,10590.000000,2979.0,0.0,26201.000000,6735.000000,6735.000000,0.0,0.0,229.000000,...,229.0,970.0,1.0,844.000000,231.000000,27.0,0.0,0.0,3.0,2.785900e+04
mean,222.048007,18.318124,1.0,NaN,30.902813,3.775056,0.419748,NaN,NaN,0.982533,...,1.0,1.0,2.9,81308.329384,59145.346320,1.0,NaN,NaN,1.0,3.283117e+06
std,95.026246,11.269409,0.0,NaN,23.740475,1.007118,0.493554,NaN,NaN,0.131291,...,0.0,0.0,NaN,23033.103291,35187.529878,0.0,NaN,NaN,0.0,1.168940e+06
min,26.000000,1.000000,1.0,NaN,1.000000,1.000000,0.000000,NaN,NaN,0.000000,...,1.0,1.0,2.9,10000.000000,10000.000000,1.0,NaN,NaN,1.0,2.683909e+06
25%,141.000000,8.839200,1.0,NaN,13.500000,3.000000,0.000000,NaN,NaN,1.000000,...,1.0,1.0,2.9,90001.000000,19000.000000,1.0,NaN,NaN,1.0,2.683936e+06
50%,248.000000,16.300000,1.0,NaN,24.130000,4.000000,0.000000,NaN,NaN,1.000000,...,1.0,1.0,2.9,90006.000000,90000.000000,1.0,NaN,NaN,1.0,2.685580e+06
75%,304.000000,26.600000,1.0,NaN,42.418000,5.000000,1.000000,NaN,NaN,1.000000,...,1.0,1.0,2.9,90006.000000,90001.000000,1.0,NaN,NaN,1.0,2.879984e+06
max,338.000000,83.000000,1.0,NaN,224.000000,5.000000,1.000000,NaN,NaN,1.000000,...,1.0,1.0,2.9,99000.000000,90013.000000,1.0,NaN,NaN,1.0,8.892957e+06


In [21]:
ofo_trees_df[ofo_trees_df.live_dead == 'L'].iloc[1]

plot_id                                                        0068
subplot_id                                                     75.0
height                                                         19.4
height_allometric                                               NaN
height_above_plot_center                                        NaN
dbh                                                            43.2
species_code                                                 CADE27
growth_form                                                    tree
live_dead                                                         L
crown_position                                                  NaN
ohvis                                                           NaN
crown_ratio                                                     NaN
crown_ratio_compacted                                           NaN
height_to_crown                                                 1.0
height_to_needle                                

In [17]:
ofo_trees_df[['height', 'dbh', 'growth_form', 'live_dead', 'crown_width_1', 'crown_width_2', 'crown_width_allometric']]

,height,dbh,growth_form,live_dead,crown_width_1,crown_width_2,crown_width_allometric
0,26.2,51.8,tree,D,NaN,NaN,NaN
1,4.5,13.9,tree,L,1.0,1.0,NaN
2,19.4,43.2,tree,L,1.0,1.0,NaN
3,23.3,67.3,tree,L,1.0,1.0,NaN
4,6.4,19.9,tree,L,1.0,1.0,NaN
...,...,...,...,...,...,...,...
34466,3.0,5.5,tree,D,NaN,NaN,NaN
34467,3.0,5.9,tree,L,NaN,NaN,NaN
34468,NaN,13.1,tree,L,NaN,NaN,NaN
34469,NaN,NaN,None,None,NaN,NaN,NaN


## FERP Trees Dataset

In [10]:
ferp_path = '../../data/ferp/geoforest/doi_10_5061_dryad_6q573n64s__v20240129/FERP123merged_20231029.csv'

# Load the dataset
ferp_trees_df = pd.read_csv(ferp_path, low_memory=False)

In [11]:
ferp_species_path = '../../data/ferp/ferp_tree_species.txt'

df_species = load_ferp_species_table(ferp_species_path)

df_species.head(3)

,Scientific name,Common name,Code,Family,Related,Genus,Specific epithet,Author
0,Acer macrophyllum Pursh,Big-leaf maple,ACERMA,Sapindaceae,,Acer,macrophyllum,Pursh
1,Adenostoma fasciculatum Hook. & Arn.,Chamise,ADENFA,Rosaceae,,Adenostoma,fasciculatum,Hook. & Arn.
2,Arbutus menziesii Pursh,Madrone,ARBUME,Ericaceae,,Arbutus,menziesii,Pursh


In [12]:
common_name_d = df_species.set_index('Code')['Common name'].to_dict()
specific_epithet_d = df_species.set_index('Code')['Specific epithet'].to_dict()
genus_d = df_species.set_index('Code')['Genus'].to_dict()

ferp_trees_df['common_name'] = ferp_trees_df['code6'].map(common_name_d)
ferp_trees_df['scientific_name'] = ferp_trees_df['code6'].map(genus_d) + ' ' + ferp_trees_df['code6'].map(specific_epithet_d)

In [13]:
ferp_trees_df

,quadrat,tag,stemtag,stemtag1,code6,east_m,north_m,east_UTM,north_UTM,dsh1_mm,...,stems1,multi2,multi3,basalarea1_m2,code6fix,locfix,notes2,notes3,common_name,scientific_name
0,E000_N000,2,1.0,NaN,QUERPA,2.6,6.7,582309.51,4096655.62,31.0,...,1.0,NaN,NaN,0.000755,NaN,loc_fixed,NaN,Tag 2 was not on the original data sheet even ...,Shreve’s oak,Quercus parvula
1,E000_N000,3,1.0,NaN,PSEUME,0.6,6.2,582307.45,4096655.65,378.0,...,1.0,NaN,NaN,0.112221,NaN,NaN,NaN,NaN,Douglas-fir,Pseudotsuga menziesii
2,E000_N000,4,1.0,NaN,QUERPA,0.5,6.8,582307.51,4096656.25,20.0,...,1.0,NaN,NaN,0.000314,NaN,NaN,NaN,NaN,Shreve’s oak,Quercus parvula
3,E000_N000,5,1.0,NaN,SEQUSE,3.1,13.7,582311.77,4096662.27,1420.0,...,1.0,NaN,NaN,1.583677,NaN,NaN,NaN,NaN,Coast redwood,Sequoia sempervirens
4,E000_N000,6,1.0,NaN,QUERPA,4.7,19.2,582314.71,4096667.18,74.0,...,1.0,multi2,NaN,0.004301,NaN,NaN,NaN,NaN,Shreve’s oak,Quercus parvula
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51011,E280_N320,34371,1.0,NaN,PSEUME,298.7,327.6,582677.22,4096891.09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Douglas-fir,Pseudotsuga menziesii
51012,E280_N300,34694,1.0,NaN,QUERPA,299.4,303.4,582671.76,4096867.51,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Shreve’s oak,Quercus parvula
51013,E320_N360,35287,1.0,NaN,SEQUSE,339.6,374.8,582728.73,4096926.40,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Coast redwood,Sequoia sempervirens
51014,E340_N040,36362,1.0,NaN,LITHDE,346.7,42.6,582651.49,4096603.23,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Tanoak,Notholithocarpus densiflorus


In [14]:
# Manual list of tree species to filter

tree_species = [
    "Shreve’s oak", "Douglas-fir", "Coast redwood", "Tanoak",
    "California hazelnut", "Coast live oak", "Madrone", "California Bay",
    "Ponderosa pine", "Knobcone pine", "Big-leaf maple", "Yellow willow",
    "Loquat", "Blue-gum eucalyptus"
]

ferp_trees_df = ferp_trees_df[ferp_trees_df['common_name'].isin(tree_species)]

len(ferp_trees_df)

41229